In [1]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS
import pandas as pd

In [4]:
df = pd.read_csv('../data/Passenger.csv')

In [14]:
df = df.rename(columns={
    'Month': 'ds',
    'Passengers': 'y'
})
df['unique_id'] = 'Serie_1'
df.head()
df['ds'] = pd.to_datetime(df['ds'])
type(df['ds'])

pandas.core.series.Series

In [5]:
sf = StatsForecast(
  models=[AutoARIMA(season_length=12)],
  freq='M',
  n_jobs=-1
)

In [6]:
season_length = 12 # Define season length as 12 months for monthly data
horizon = 1 # Forecast horizon is set to 1 month

# Define a list of models for forecasting
models = [
    AutoARIMA(season_length=season_length), # ARIMA model with automatic order selection and seasonal component
    AutoETS(season_length=season_length), # ETS model with automatic error, trend, and seasonal component
]

# Instantiate StatsForecast class with models, data frequency ('M' for monthly),
# and parallel computation on all CPU cores (n_jobs=-1)
sf = StatsForecast(
    models=models, # models for forecasting
    freq=pd.offsets.MonthEnd(),  # frequency of the timestamps
    n_jobs=-1  # number of jobs to run in parallel, -1 means using all processors
)

In [16]:
# Generate forecasts for the specified horizon using the sf object
Y_hat_df = sf.forecast(df=df, h=12) # forecast data
# Display the first few rows of the forecast DataFrame
Y_hat_df.head() # preview of forecasted data

,unique_id,ds,AutoARIMA,AutoETS
0,Serie_1,1960-12-31,444.309570,442.357178
1,Serie_1,1961-01-31,418.213745,428.267365
2,Serie_1,1961-02-28,446.243408,492.974792
3,Serie_1,1961-03-31,488.234222,477.369995
4,Serie_1,1961-04-30,499.237061,477.602814


In [17]:
sf.fit(df=df) # Fit the models to the data using the fit method of the StatsForecast object
sf.fitted_ # Access fitted models from the StatsForecast object
Y_hat_df = sf.predict(h=horizon) # Predict or forecast 'horizon' steps ahead using the predict method
Y_hat_df.head() # Preview the first few rows of the forecasted data

,unique_id,ds,AutoARIMA,AutoETS
0,Serie_1,1960-12-31,444.30957,442.357178


In [20]:
from statsforecast.utils import ConformalIntervals

cv = sf.cross_validation(
    df=df,
    h=6,
    step_size=1,
    n_windows=3
)